In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


StatementMeta(, 0ddbe81a-1f20-4a23-aed4-933cbc90b470, 3, Finished, Available, Finished, False)

# silver_telemetry
1. fact_telemetry
2. fact_energy
3. dim_date
# silver_events
1. fact_event
# silver_asset_metadata
1. - dim_site
2. - dim_building
3. - dim_asset

In [2]:
display(
    spark.sql("""
        CREATE OR REPLACE TABLE dbo.gold_dim_site AS
        SELECT DISTINCT
            site_id
        FROM dbo.silver_asset_metadata
        WHERE site_id IS NOT NULL
          AND TRIM(site_id) <> ''
          Order by site_id ASC
    """)
)


StatementMeta(, 0ddbe81a-1f20-4a23-aed4-933cbc90b470, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 24e6d04b-7dc1-4aab-9d38-98acd8bbb53d)

# we  build the building dimension from the telemetry data.

In [3]:
display(
    spark.sql("""
        CREATE OR REPLACE TABLE dbo.gold_dim_building AS

        SELECT DISTINCT
            building_id,
            site_id
            

        FROM dbo.silver_telemetry

        WHERE site_id IS NOT NULL
          AND building_id IS NOT NULL
          AND TRIM(site_id) <> ''
          AND TRIM(building_id) <> ''
          Order by building_id ASC
    """)
)

StatementMeta(, 0ddbe81a-1f20-4a23-aed4-933cbc90b470, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3c3d661a-aa3f-41b6-bf7b-a387915fba11)

# asset metadata is the best source for the asset dimension.

In [4]:
display(
    spark.sql("""
        CREATE OR REPLACE TABLE dbo.gold_dim_asset AS

        SELECT DISTINCT
            asset_id,
            asset_name,
            asset_type,
            manufacturer,
            installation_date,
            site_id

        FROM dbo.silver_asset_metadata

        WHERE asset_id IS NOT NULL
          AND TRIM(asset_id) <> ''
          Order by asset_id ASC
    """)
)

StatementMeta(, 0ddbe81a-1f20-4a23-aed4-933cbc90b470, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 339f4ab3-8a25-4599-80de-b6e4ac3d24d7)

# We  derive it from telemetry timestamps.

In [5]:
display(
    spark.sql("""
        CREATE OR REPLACE TABLE dbo.gold_dim_time AS

        SELECT DISTINCT
            DATE(timestamp) AS date_key,
            YEAR(timestamp) AS year,
            MONTH(timestamp) AS month,
            DAY(timestamp) AS day,
            DAYOFWEEK(timestamp) AS day_of_week,
            WEEKOFYEAR(timestamp) AS week_of_year,
            DAYOFYEAR(timestamp) AS day_of_year

        FROM dbo.silver_telemetry

        WHERE timestamp IS NOT NULL
    Order by date_key ASC
    """)
)

StatementMeta(, 0ddbe81a-1f20-4a23-aed4-933cbc90b470, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 507d573c-a30d-4c34-ad10-81e5b8506e8b)

# This is the main IoT telemetry fact.

In [6]:
display(
    spark.sql("""
        CREATE OR REPLACE TABLE dbo.gold_fact_telemetry
        USING DELTA
        PARTITIONED BY (date_key)
        AS

        SELECT
            ROW_NUMBER() OVER (
                ORDER BY
                    timestamp,
                    site_id,
                    building_id,
                    asset_id,
                    sensor_id
            ) AS telemetry_key,

            DATE(timestamp) AS date_key,
            timestamp,
            site_id,
            building_id,
            asset_id,
            sensor_id,

            temperature,
            humidity,
            pressure,
            vibration,
            power_consumption,
            operating_mode,

            source_file,
            ingestion_timestamp

        FROM dbo.dq_telemetry

        WHERE validation_status = 'VALID'
          AND timestamp IS NOT NULL
    """)
)

StatementMeta(, 0ddbe81a-1f20-4a23-aed4-933cbc90b470, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 30158611-0c88-47ba-a089-ca54d0f6c73c)

In [7]:
display(
    spark.sql("""
        CREATE OR REPLACE TABLE dbo.gold_fact_energy
        USING DELTA
        PARTITIONED BY (date_key)
        AS

        SELECT
            ROW_NUMBER() OVER (
                ORDER BY
                    DATE_TRUNC('hour', timestamp),
                    site_id,
                    building_id,
                    asset_id
            ) AS energy_key,

            DATE(timestamp) AS date_key,

            DATE_TRUNC('hour', timestamp) AS hour,

            site_id,
            building_id,
            asset_id,

            ROUND(
                SUM(power_consumption),
                2
            ) AS hourly_energy_consumption,

            ROUND(
                AVG(power_consumption),
                2
            ) AS avg_power_consumption,

            COUNT(*) AS reading_count

        FROM dbo.dq_telemetry

        WHERE validation_status = 'VALID'
          AND timestamp IS NOT NULL
          AND power_consumption IS NOT NULL

        GROUP BY
            DATE_TRUNC('hour', timestamp),
            DATE(timestamp),
            site_id,
            building_id,
            asset_id
    """)
)

StatementMeta(, 0ddbe81a-1f20-4a23-aed4-933cbc90b470, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 95bb2346-8b01-489a-a5a9-b94e66f52e4e)

# events become an event fact.

In [8]:
display(
    spark.sql("""
        CREATE OR REPLACE TABLE dbo.gold_fact_event
        USING DELTA
        PARTITIONED BY (date_key)
        AS

        SELECT
            ROW_NUMBER() OVER (
                ORDER BY
                    timestamp,
                    event_id,
                    asset_id
            ) AS event_key,

            DATE(timestamp) AS date_key,
            timestamp,
            event_id,
            asset_id,
            event_type,
            severity,
            message,

            source_file,
            ingestion_timestamp

        FROM dbo.dq_events

        WHERE validation_status = 'VALID'
          AND timestamp IS NOT NULL
    """)
)

StatementMeta(, 0ddbe81a-1f20-4a23-aed4-933cbc90b470, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 67ef4ce8-de8b-4129-b865-ead045ba87a0)

In [9]:
display(
    spark.sql("""
        SELECT 'gold_dim_asset' AS table_name, COUNT(*) AS record_count
        FROM dbo.gold_dim_asset

        UNION ALL

        SELECT 'gold_dim_building', COUNT(*)
        FROM dbo.gold_dim_building

        UNION ALL

        SELECT 'gold_dim_site', COUNT(*)
        FROM dbo.gold_dim_site

        UNION ALL

        SELECT 'gold_dim_time', COUNT(*)
        FROM dbo.gold_dim_time

        UNION ALL

        SELECT 'gold_fact_energy', COUNT(*)
        FROM dbo.gold_fact_energy

        UNION ALL

        SELECT 'gold_fact_event', COUNT(*)
        FROM dbo.gold_fact_event

        UNION ALL

        SELECT 'gold_fact_telemetry', COUNT(*)
        FROM dbo.gold_fact_telemetry
    """)
)

StatementMeta(, 0ddbe81a-1f20-4a23-aed4-933cbc90b470, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 15abb6e7-4aac-4414-8044-e08060cc591f)

In [10]:
display(spark.sql("SELECT * FROM dbo.gold_dim_asset"))
display(spark.sql("SELECT * FROM dbo.gold_dim_building"))
display(spark.sql("SELECT * FROM dbo.gold_dim_site"))
display(spark.sql("SELECT * FROM dbo.gold_dim_time"))
display(spark.sql("SELECT * FROM dbo.gold_fact_energy"))
display(spark.sql("SELECT * FROM dbo.gold_fact_event"))
display(spark.sql("SELECT * FROM dbo.gold_fact_telemetry"))

StatementMeta(, 0ddbe81a-1f20-4a23-aed4-933cbc90b470, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3eec2c0a-ea64-4628-b5af-f8ec370ba729)

SynapseWidget(Synapse.DataFrame, 0865d9af-43ce-4a67-9da6-50afb0971529)

SynapseWidget(Synapse.DataFrame, 6b9570b6-0418-428b-8d8d-42a68bbe4a03)

SynapseWidget(Synapse.DataFrame, 784da639-6356-48e6-babc-f2a668290e30)

SynapseWidget(Synapse.DataFrame, 8bc20f27-8716-43a8-b526-996d1d37f290)

SynapseWidget(Synapse.DataFrame, 5090dc79-f41f-49ce-a09c-2d9f59a5b809)

SynapseWidget(Synapse.DataFrame, 6b912a89-cf2d-4303-ad0d-e45ef4789a64)

# ------------------------------------------------------

                   
# Dashboarding	 =                    Facts + dimensions + aggregated metrics
# Historical reporting	=             Timestamped fact tables + dim_time
# Machine learning	   =              Detailed fact_telemetry + historical events

# -------------------------------------------------------------------------

# This main hierarchy 

In [11]:
display(
    spark.sql("""
        CREATE OR REPLACE TABLE dbo.gold_asset_hierarchy AS

        SELECT
            a.site_id,
            b.building_id,
            a.asset_name,
            a.asset_id,
            a.asset_type

        FROM dbo.gold_dim_asset a

        INNER JOIN (
            SELECT
                building_id,
                site_id,
                ROW_NUMBER() OVER (
                    PARTITION BY site_id
                    ORDER BY building_id
                ) AS building_number
            FROM dbo.gold_dim_building
        ) b
            ON a.site_id = b.site_id

        WHERE
            (
                b.building_number = 1
                AND CAST(SUBSTRING(a.asset_id, 2) AS INT)
                    BETWEEN
                    CASE
                        WHEN a.site_id = 'S001' THEN 1
                        WHEN a.site_id = 'S002' THEN 11
                        WHEN a.site_id = 'S003' THEN 21
                        WHEN a.site_id = 'S004' THEN 31
                        WHEN a.site_id = 'S005' THEN 41
                    END
                    AND
                    CASE
                        WHEN a.site_id = 'S001' THEN 5
                        WHEN a.site_id = 'S002' THEN 15
                        WHEN a.site_id = 'S003' THEN 25
                        WHEN a.site_id = 'S004' THEN 35
                        WHEN a.site_id = 'S005' THEN 45
                    END
            )
            OR
            (
                b.building_number = 2
                AND CAST(SUBSTRING(a.asset_id, 2) AS INT)
                    BETWEEN
                    CASE
                        WHEN a.site_id = 'S001' THEN 6
                        WHEN a.site_id = 'S002' THEN 16
                        WHEN a.site_id = 'S003' THEN 26
                        WHEN a.site_id = 'S004' THEN 36
                        WHEN a.site_id = 'S005' THEN 46
                    END
                    AND
                    CASE
                        WHEN a.site_id = 'S001' THEN 10
                        WHEN a.site_id = 'S002' THEN 20
                        WHEN a.site_id = 'S003' THEN 30
                        WHEN a.site_id = 'S004' THEN 40
                        WHEN a.site_id = 'S005' THEN 50
                    END
            )
    """)
)

StatementMeta(, 0ddbe81a-1f20-4a23-aed4-933cbc90b470, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 78a9d1ab-c672-4e02-ac54-e3ffdee3fcbe)

In [12]:
display(spark.sql("SELECT * FROM dbo.gold_asset_hierarchy"))

StatementMeta(, 0ddbe81a-1f20-4a23-aed4-933cbc90b470, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f4ad0698-ef94-4ef5-8191-b24ad25b7cea)

# verify  current hierarchy

In [13]:
display(
    spark.sql("""
        SELECT
            site_id,
            building_id,
            COUNT(*) AS asset_count
        FROM dbo.gold_asset_hierarchy
        GROUP BY site_id, building_id
        ORDER BY site_id, building_id
    """)
)

StatementMeta(, 0ddbe81a-1f20-4a23-aed4-933cbc90b470, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d0220f47-1435-43bd-8b6c-25a1ca3b078a)

# check every site

In [14]:
display(
    spark.sql("""
        SELECT
            site_id,
            COUNT(DISTINCT building_id) AS building_count,
            COUNT(DISTINCT asset_id) AS asset_count
        FROM dbo.gold_asset_hierarchy
        GROUP BY site_id
        ORDER BY site_id
    """)
)

StatementMeta(, 0ddbe81a-1f20-4a23-aed4-933cbc90b470, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e6d4f694-0e70-4ff6-969f-97553b19fc3a)

In [15]:
display(
    spark.sql("""
        CREATE OR REPLACE TABLE dbo.gold_asset_relationship AS

        SELECT
            site_id,
            building_id,
            parent_asset_id,
            child_asset_id,
            'PARENT_CHILD' AS relationship_type

        FROM VALUES

            -- S001 / B001
            ('S001', 'B001', 'A001', 'A002'),
            ('S001', 'B001', 'A001', 'A005'),

            -- S001 / B002
            ('S001', 'B002', 'A006', 'A007'),
            ('S001', 'B002', 'A009', 'A010'),

            -- S002 / B003
            ('S002', 'B003', 'A011', 'A012'),
            ('S002', 'B003', 'A011', 'A015'),

            -- S002 / B004
            ('S002', 'B004', 'A016', 'A017'),
            ('S002', 'B004', 'A019', 'A020'),

            -- S003 / B005
            ('S003', 'B005', 'A021', 'A022'),
            ('S003', 'B005', 'A021', 'A025'),

            -- S003 / B006
            ('S003', 'B006', 'A026', 'A027'),
            ('S003', 'B006', 'A029', 'A030'),

            -- S004 / B007
            ('S004', 'B007', 'A031', 'A032'),
            ('S004', 'B007', 'A031', 'A035'),

            -- S004 / B008
            ('S004', 'B008', 'A036', 'A037'),
            ('S004', 'B008', 'A039', 'A040'),

            -- S005 / B009
            ('S005', 'B009', 'A041', 'A042'),
            ('S005', 'B009', 'A041', 'A045'),

            -- S005 / B010
            ('S005', 'B010', 'A046', 'A047'),
            ('S005', 'B010', 'A049', 'A050')

        AS t(
            site_id,
            building_id,
            parent_asset_id,
            child_asset_id
        )
    """)
)

StatementMeta(, 0ddbe81a-1f20-4a23-aed4-933cbc90b470, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, da6e4412-1759-4e8e-b7c6-3ac74d29d252)

# Retrieve all assets under a site.

In [16]:
display(
    spark.sql("""
        SELECT
            site_id,
            building_id,
            asset_id,
            asset_name,
            asset_type

        FROM dbo.gold_asset_hierarchy

        WHERE site_id = 'S001'

        ORDER BY
            building_id,
            asset_id
    """)
)

StatementMeta(, 0ddbe81a-1f20-4a23-aed4-933cbc90b470, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4148a5a8-2723-4ea1-bd73-4b63ab49c2cf)

# Retrieve Parent and Child Assets.

In [17]:
display(
    spark.sql("""
        SELECT
            r.site_id,
            r.building_id,

            r.parent_asset_id,
            p.asset_name AS parent_asset_name,
            p.asset_type AS parent_asset_type,

            r.child_asset_id,
            c.asset_name AS child_asset_name,
            c.asset_type AS child_asset_type,

            r.relationship_type

        FROM dbo.gold_asset_relationship r

        LEFT JOIN dbo.gold_dim_asset p
            ON r.parent_asset_id = p.asset_id

        LEFT JOIN dbo.gold_dim_asset c
            ON r.child_asset_id = c.asset_id

        ORDER BY
            r.site_id,
            r.building_id,
            r.parent_asset_id,
            r.child_asset_id
    """)
)

StatementMeta(, 0ddbe81a-1f20-4a23-aed4-933cbc90b470, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d956e00e-33b4-4d99-8ca7-ae98e9cd338d)

# Find downstream impacted assets

If Chiller-01 fails, the query identifies its downstream child assets AHU-02 and AHU-05.

In [18]:
display(
    spark.sql("""
        SELECT
            r.site_id,
            r.building_id,

            r.parent_asset_id,
            p.asset_name AS parent_asset_name,

            r.child_asset_id,
            c.asset_name AS impacted_asset_name,
            c.asset_type AS impacted_asset_type

        FROM dbo.gold_asset_relationship r

        LEFT JOIN dbo.gold_dim_asset p
            ON r.parent_asset_id = p.asset_id

        LEFT JOIN dbo.gold_dim_asset c
            ON r.child_asset_id = c.asset_id

        WHERE r.parent_asset_id = 'A001'

        ORDER BY r.child_asset_id
    """)
)

StatementMeta(, 0ddbe81a-1f20-4a23-aed4-933cbc90b470, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4d5eb39e-b83f-431f-8651-642970c4a0ef)

# Identify orphan assets

- An orphan asset is an asset that is expected to have a parent based on the hierarchy rules, but no valid parent relationship exists. For example, an AHU without its expected Chiller or Pump parent.
- All AHUs currently have a parent relationship.

Normally:

    Chiller-01
     └── AHU-02

Suppose data like that:

Building B001

 └──Chiller-01
 
└──AHU-02

Expected:

Chiller-01 (A001)

       └── AHU-02 (A002)


Actual:

Chiller-01 (A001)

AHU-02 (A002)

     Wrong
     
  No parent


just example for this:
- AHU-02 is an orphan asset because, based on our hierarchy rule, an AHU is expected to have a parent asset, but no parent relationship exists for it.

In [19]:
display(
    spark.sql("""
        SELECT
            a.site_id,
            a.asset_id,
            a.asset_name,
            a.asset_type

        FROM dbo.gold_dim_asset a

        LEFT JOIN dbo.gold_asset_relationship r
            ON a.asset_id = r.child_asset_id

        WHERE a.asset_type = 'AHU'
          AND r.child_asset_id IS NULL

        ORDER BY
            a.site_id,
            a.asset_id
    """)
)

StatementMeta(, 0ddbe81a-1f20-4a23-aed4-933cbc90b470, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0b27ff65-e713-4c62-a441-b310b5e33e25)

# Identify disconnected assets

A disconnected asset is an asset that exists in the asset master but has no relationship with any other asset in the hierarchy. 

For example, A008 Pump-08 is disconnected because A008 does not appear as either a parent or a child in the asset relationship table.

Has parent OR has child

        ↓

     CONNECTED

Has NO parent AND NO child

        ↓
        
    DISCONNECTED

In [20]:
display(
    spark.sql("""
        SELECT
            a.site_id,
            a.asset_id,
            a.asset_name,
            a.asset_type

        FROM dbo.gold_dim_asset a

        LEFT JOIN dbo.gold_asset_relationship parent_rel
            ON a.asset_id = parent_rel.child_asset_id

        LEFT JOIN dbo.gold_asset_relationship child_rel
            ON a.asset_id = child_rel.parent_asset_id

        WHERE parent_rel.child_asset_id IS NULL
          AND child_rel.parent_asset_id IS NULL

        ORDER BY
            a.site_id,
            a.asset_id
    """)
)

StatementMeta(, 0ddbe81a-1f20-4a23-aed4-933cbc90b470, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1f4925e5-2134-4d20-a7ea-4c9d50b63198)

# **# What is partitioning?**

### **Partitioning physically organizes a large Delta table into separate folders based on a column.**

In [21]:
display(
    spark.sql("""
        DESCRIBE DETAIL dbo.gold_fact_telemetry
    """)
)

StatementMeta(, 0ddbe81a-1f20-4a23-aed4-933cbc90b470, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7c9b9c32-9984-4307-b5b0-41744d63ec64)

In [22]:
display(
    spark.sql("""
        SELECT
            date_key,
            COUNT(*) AS record_count
        FROM dbo.gold_fact_telemetry
        GROUP BY date_key
        ORDER BY date_key
    """)
)

StatementMeta(, 0ddbe81a-1f20-4a23-aed4-933cbc90b470, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 56dd4686-3940-4f6d-9ccd-11ea8791cc09)

In [23]:
display(
    spark.sql("""
        OPTIMIZE dbo.gold_fact_telemetry
    """)
)

StatementMeta(, 0ddbe81a-1f20-4a23-aed4-933cbc90b470, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ec76e439-5ead-4c36-9cd4-3e233f0dc405)

In [24]:
display(
    spark.sql("""
        OPTIMIZE dbo.gold_fact_energy
    """)
)

StatementMeta(, 0ddbe81a-1f20-4a23-aed4-933cbc90b470, 26, Finished, Available, Finished, True)

SynapseWidget(Synapse.DataFrame, d84e4db3-8d63-4e85-8e50-b502e9a7e37b)

In [25]:
display(
    spark.sql("""
        OPTIMIZE dbo.gold_fact_event
    """)
)

StatementMeta(, 0ddbe81a-1f20-4a23-aed4-933cbc90b470, 27, Finished, Available, Finished, True)

SynapseWidget(Synapse.DataFrame, 37bec7e8-2b9e-4407-9ec7-d2a95caff8dd)

## Indexing strategy 

### Because our Gold layer is implemented as Delta tables in Microsoft Fabric Lakehouse, we don't use traditional clustered or non-clustered indexes like SQL Server. Instead, we optimize data access through date-based partitioning and Delta table optimization. The high-volume fact tables are partitioned by date_key, while the smaller dimension tables don't require additional indexing.

In [26]:
# CREATE INDEX IX_Asset
# ON FactTelemetry(asset_id);

StatementMeta(, 0ddbe81a-1f20-4a23-aed4-933cbc90b470, 28, Finished, Available, Finished, True)

In [27]:
# WHERE asset_id = 'A001'

StatementMeta(, 0ddbe81a-1f20-4a23-aed4-933cbc90b470, 29, Finished, Available, Finished, True)